# V2 — CNN-VAE: Better Text-to-Pixel Generation
### Same VAE idea, but now we use CNNs to preserve image structure

We already know that CNNs are good at learning **spatial patterns** in images. In V1 we flattened the image immediately, which threw away much of that spatial structure.

In V2:

```text
IMAGE + WORD → CNN ENCODER → latent z → CNN DECODER + WORD → IMAGE
```

Five words: **heart · face · robot · tree · spaceship**


## 1. Create 10,000 training images

We use 2,000 slightly different examples for each of the five objects.


In [ ]:
# Download our reproducible dataset generator.
!wget -q -O generate_dataset.py https://raw.githubusercontent.com/dujing82-blip/tiny-black-white-generative-ai/main/generate_dataset.py

# Make a fresh copy of the dataset.
!rm -rf pixel_dataset
!python generate_dataset.py


## 2. Load the black-and-white images


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from PIL import Image

table = pd.read_csv('pixel_dataset/labels.csv')
words = ['heart', 'face', 'robot', 'tree', 'spaceship']

images = []
for filename in table['filename']:
    img = Image.open(os.path.join('pixel_dataset', filename)).convert('L')
    images.append(np.array(img, dtype=np.float32) / 255.0)

# CNNs expect a channel dimension.
# (10000, 16, 16) becomes (10000, 16, 16, 1).
X = np.expand_dims(np.stack(images), axis=-1)
y = table['label'].to_numpy(np.int32)

print('Images:', X.shape)
print('Labels:', y.shape)


### Look at one example of each object


In [ ]:
plt.figure(figsize=(10,2))
for i, word in enumerate(words):
    index = np.where(y == i)[0][0]
    plt.subplot(1,5,i+1)
    plt.imshow(X[index,:,:,0], cmap='gray', vmin=0, vmax=1, interpolation='nearest')
    plt.title(word)
    plt.axis('off')
plt.show()


# 3. CNN Encoder

The encoder's job is to **compress** an image.

```text
16×16 image
    ↓ Conv2D
 8×8×32
    ↓ Conv2D
 4×4×64
    ↓
 latent z
```

The word embedding tells the encoder which object the image represents.


In [ ]:
LATENT_DIM = 8
WORD_DIM = 8

# ----- TWO INPUTS -----
image_input = keras.Input(shape=(16,16,1), name='image')
word_input = keras.Input(shape=(), dtype='int32', name='word')

# ----- CNN IMAGE PATH -----
# stride=2 cuts width and height in half: 16x16 -> 8x8.
x = layers.Conv2D(32, 3, strides=2, padding='same', activation='relu')(image_input)

# Another stride=2: 8x8 -> 4x4.
x = layers.Conv2D(64, 3, strides=2, padding='same', activation='relu')(x)

# Convert the 4x4x64 feature map into one vector.
x = layers.Flatten()(x)

# ----- WORD PATH -----
# Turn heart/face/robot/tree/spaceship into a learned vector.
word_vector = layers.Embedding(input_dim=5, output_dim=WORD_DIM)(word_input)
word_vector = layers.Flatten()(word_vector)

# Combine visual features with the word representation.
x = layers.Concatenate()([x, word_vector])
x = layers.Dense(128, activation='relu')(x)

# A VAE learns a DISTRIBUTION in latent space, not one exact point.
z_mean = layers.Dense(LATENT_DIM, name='z_mean')(x)
z_log_var = layers.Dense(LATENT_DIM, name='z_log_var')(x)

class Sampling(layers.Layer):
    def call(self, inputs):
        mean, log_var = inputs
        noise = tf.random.normal(shape=tf.shape(mean))
        return mean + tf.exp(0.5 * log_var) * noise

z = Sampling()([z_mean, z_log_var])

encoder = keras.Model([image_input, word_input], [z_mean, z_log_var, z], name='CNN_encoder')
encoder.summary()


# 4. CNN Decoder / Generator

Now we go in the opposite direction:

```text
z + word
   ↓ Dense
 4×4×64
   ↓ Conv2DTranspose
 8×8×64
   ↓ Conv2DTranspose
16×16×32
   ↓
16×16 image
```

A useful mental model: **the encoder shrinks; the decoder grows.**


In [ ]:
z_input = keras.Input(shape=(LATENT_DIM,), name='z')
decoder_word_input = keras.Input(shape=(), dtype='int32', name='word')

# Convert the requested word into numbers.
decoder_word_vector = layers.Embedding(input_dim=5, output_dim=WORD_DIM)(decoder_word_input)
decoder_word_vector = layers.Flatten()(decoder_word_vector)

# WHAT to draw (word) + HOW it varies (z).
x = layers.Concatenate()([z_input, decoder_word_vector])

# Create enough numbers for a 4x4x64 feature map.
x = layers.Dense(4*4*64, activation='relu')(x)
x = layers.Reshape((4,4,64))(x)

# Grow 4x4 -> 8x8.
x = layers.Conv2DTranspose(64, 3, strides=2, padding='same', activation='relu')(x)

# Grow 8x8 -> 16x16.
x = layers.Conv2DTranspose(32, 3, strides=2, padding='same', activation='relu')(x)

# Produce ONE black/white probability for every pixel.
# sigmoid keeps each output between 0 and 1.
output_image = layers.Conv2D(1, 3, padding='same', activation='sigmoid')(x)

decoder = keras.Model([z_input, decoder_word_input], output_image, name='CNN_decoder')
decoder.summary()


# 5. The two VAE losses

A VAE has two goals:

### Reconstruction loss
**“Can I draw the training image correctly?”**

Because every pixel is black or white, we use Keras's built-in **BinaryCrossentropy**.

### KL loss
**“Is the latent space organized so that random sampling works?”**

We hide its formula inside one small helper function. For this lesson, the important idea is its purpose, not memorizing the formula.


In [ ]:
# Built-in Keras loss for black/white pixels.
reconstruction_loss_fn = keras.losses.BinaryCrossentropy(reduction='none')

# The special regularization term that makes this a VARIATIONAL autoencoder.
def kl_loss(z_mean, z_log_var):
    return -0.5 * tf.reduce_mean(
        tf.reduce_sum(
            1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var),
            axis=1
        )
    )

# Small beta: for this classroom demo we prioritize drawing recognizable images.
BETA = 0.005


# 6. Train

Notice how short the important part now is:

```text
encode → decode → reconstruction loss + KL loss → learn
```


In [ ]:
class VAE(keras.Model):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def train_step(self, data):
        images, word_ids = data

        with tf.GradientTape() as tape:
            # STEP 1: Image -> latent z
            z_mean, z_log_var, z = self.encoder([images, word_ids], training=True)

            # STEP 2: latent z + word -> reconstructed image
            reconstructed = self.decoder([z, word_ids], training=True)

            # GOAL 1: Make the reconstructed pixels match the real pixels.
            pixel_loss = reconstruction_loss_fn(images, reconstructed)
            reconstruction_loss = tf.reduce_mean(tf.reduce_sum(pixel_loss, axis=[1,2]))

            # GOAL 2: Keep latent space organized.
            latent_loss = kl_loss(z_mean, z_log_var)

            # Total VAE loss.
            total_loss = reconstruction_loss + BETA * latent_loss

        # Standard backpropagation — same idea as our previous neural networks.
        gradients = tape.gradient(total_loss, self.trainable_weights)
        self.optimizer.apply_gradients(zip(gradients, self.trainable_weights))

        return {'loss': total_loss,
                'reconstruction': reconstruction_loss,
                'KL': latent_loss}

dataset = tf.data.Dataset.from_tensor_slices((X,y))
dataset = dataset.shuffle(len(X)).batch(128).prefetch(tf.data.AUTOTUNE)

vae = VAE(encoder, decoder)
vae.compile(optimizer=keras.optimizers.Adam(0.001))

# More data + CNN usually needs only a moderate number of epochs.
history = vae.fit(dataset, epochs=40)


# 7. Generate new images

Generation uses only the **decoder**:

```text
word + random z → decoder → new image
```


In [ ]:
word_to_id = {word:i for i,word in enumerate(words)}

def generate(word, number=10, binary=True):
    if word not in word_to_id:
        print('Choose one of:', words)
        return

    # New random latent points = new variations.
    random_z = np.random.normal(size=(number, LATENT_DIM)).astype('float32')
    word_ids = np.full(number, word_to_id[word], dtype=np.int32)

    generated = decoder.predict([random_z, word_ids], verbose=0)

    plt.figure(figsize=(15,2))
    for i in range(number):
        image = generated[i,:,:,0]

        # The neural network outputs probabilities between 0 and 1.
        # For pixel art, thresholding makes the final display truly black/white.
        if binary:
            image = image > 0.5

        plt.subplot(1,number,i+1)
        plt.imshow(image, cmap='gray', vmin=0, vmax=1, interpolation='nearest')
        plt.axis('off')
    plt.suptitle('Prompt: ' + word, fontsize=16)
    plt.show()

generate('robot')


## 8. Same word, different z

Run each line again. The **word stays fixed**, while random `z` creates variations.


In [ ]:
generate('heart')
generate('robot')
generate('spaceship')


## 9. See the raw probabilities vs. black/white pixels

The decoder actually predicts probabilities. Thresholding is only for display; **we never threshold during training**.


In [ ]:
# Soft probability image
generate('tree', number=8, binary=False)

# Exactly the same idea, displayed as black/white pixel art
generate('tree', number=8, binary=True)


## 10. Interactive generator


In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

box = widgets.Dropdown(options=words, value='robot', description='Word:')
button = widgets.Button(description='Generate')
output = widgets.Output()

def clicked(_):
    with output:
        clear_output(wait=True)
        generate(box.value)

button.on_click(clicked)
display(box, button, output)


# V1 vs. V2 — what changed?

| V1 | V2 |
|---|---|
| Flatten image immediately | CNN preserves spatial patterns |
| Dense encoder/decoder | CNN encoder + CNN decoder |
| 5,000 images | 10,000 images |
| latent = 4 | latent = 8 |
| MSE | Binary Crossentropy |
| β = 0.02 | β = 0.005 |

### The VAE idea did **not** change.

```text
                 TRAINING
image + word → encoder → z → decoder → reconstructed image

                GENERATION
random z + word ─────────→ decoder → brand-new image
```

CNN simply gives the VAE a better way to understand and generate spatial image structure.
